In [1]:
from pathlib import Path
import sys

# Project root
PROJECT_ROOT = Path.cwd().parents[2]
sys.path.append(str(PROJECT_ROOT))

import os
import xarray as xr
from modules import read_files as read
import numpy as np
import cdsapi
import os

### Configuration

In [2]:
output_folder = '../../../data/preprocessing/CERRA/download' # folder for downloadeded .nc data

mohid_dir = '../../../mohid/convert2hdf5' # folder of MOHID convert2hdf5 tool

m_dat = r'convtohdf.dat'  # configuration file for using MOHID convert2hdf5 tool  

outdir = '../../../data/preprocessing/CERRA/conversion' # folder for converted .hdf5 data

# when downloading from the API the full months are selected, choose start and end to subset time
start = '2023-01-08T00:00:00.000000000'
end = '2023-02-23T21:00:00.000000000'

# when downloading from the API, it is not possible to select a region due to the projection
# choose boundaries to subset afterwards
lat_min = 27
lat_max = 30
lon_min = 342 # corresponds to -18 
lon_max = 345 # corresponds to -15

# API request
request = {
    "variable": [
        "10m_wind_direction",
        "10m_wind_speed",
        "2m_relative_humidity",
        "2m_temperature",
        "land_sea_mask",
        "surface_pressure",
        "total_cloud_cover"
    ],
    "level_type": "surface_or_atmosphere",
    "data_type": ["reanalysis"],
    "product_type": "analysis",
    "year": ["2023"],
    "month": ["01", "02"],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "03:00", "06:00",
        "09:00", "12:00", "15:00",
        "18:00", "21:00"
    ],
    "data_format": "netcdf"
}

### Download data through API
(Remember to create .cdsapirc file with credentials)

In [ ]:
output_file = os.path.join(
    output_folder,
    "cerra.nc"
)

dataset = "reanalysis-cerra-single-levels"

client = cdsapi.Client()
client.retrieve(dataset, request).download(output_file)

### Subset

In [3]:
ds = xr.open_dataset('../../../data/preprocessing/CERRA/download/cerra.nc', decode_coords="all")

lat = ds["latitude"]
lon = ds["longitude"]

mask = (
    (lat >= lat_min) &
    (lat <= lat_max) &
    (lon >= lon_min) &
    (lon <= lon_max)
)

ds_subset = ds.where(mask, drop=True)
ds_subset = ds_subset.sel(valid_time=slice(start,end))
ds_subset['t2m'] = ds_subset['t2m'] - 273.15 #convert from kelvin to celsius degrees

#compute wind components
ds_subset['u10'] = - ds_subset['si10'] * np.sin(ds_subset['wdir10'] * np.pi / 180)
ds_subset['v10'] = - ds_subset['si10'] * np.cos(ds_subset['wdir10'] * np.pi / 180)

ds_subset['r2'] = ds_subset['r2']/100
ds_subset['tcc'] = ds_subset['tcc']/100

# save
ds_subset.to_netcdf('../../../data/preprocessing/CERRA/download/cerra_subset.nc')
ds_subset

<xarray.Dataset> Size: 79MB
Dimensions:     (valid_time: 376, y: 79, x: 74)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 3kB 2023-01-08 ... 2023-02-23T21:...
    latitude    (y, x) float64 47kB 26.23 26.25 26.27 ... 30.72 30.74 30.75
    longitude   (y, x) float64 47kB 342.3 342.4 342.4 ... 344.6 344.6 344.7
    expver      (valid_time) <U4 6kB ...
Dimensions without coordinates: y, x
Data variables:
    wdir10      (valid_time, y, x) float32 9MB nan nan nan nan ... nan nan nan
    si10        (valid_time, y, x) float32 9MB nan nan nan nan ... nan nan nan
    r2          (valid_time, y, x) float32 9MB nan nan nan nan ... nan nan nan
    t2m         (valid_time, y, x) float32 9MB nan nan nan nan ... nan nan nan
    lsm         (valid_time, y, x) float32 9MB nan nan nan nan ... nan nan nan
    sp          (valid_time, y, x) float32 9MB nan nan nan nan ... nan nan nan
    tcc         (valid_time, y, x) float32 9MB nan nan nan nan ... nan nan nan
    u10         (valid_time, y, x) float32 9MB nan nan nan nan ... nan nan nan
    v10         (valid_time, y, x) float32 9MB nan nan nan nan ... nan nan nan
Attributes:
    GRIB_centre:             eswi
    GRIB_centreDescription:  Norrkoping
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Norrkoping
    history:                 2026-08-05T09:40 GRIB to CDM+CF via cfgrib-0.9.1...

### Conversion to hdf5

In [4]:
read.convert2hdf5(mohid_dir, m_dat, outdir)

Running MOHID Convert2Hdf5...
Convert2Hdf5.exe > c:\Users\karoa\MOHID_internship\MOHID_model_workflow\data\preprocessing\CERRA\conversion\mohidlog.dat


0